<a href="https://colab.research.google.com/github/gabrielramirez2109/proyectomatesegundo/blob/main/limpiezadedatos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import os
# Lista todos los elementos en /content
print(os.listdir('/content'))


['.config', '.ipynb_checkpoints', 'conjunto_de_datos_poblacion_enigh2022_ns.csv', 'conjunto_de_datos_concentradohogar_enigh2022_ns.csv', 'conjunto_de_datos_ingresos_enigh2022_ns.csv', 'sample_data']


In [8]:
path_prefix = '/content/'


In [9]:
import pandas as pd
import os

# 1. Ajusta el prefijo según tu entorno en Colab
path_prefix = '/content/'

# 2. Rutas de los CSV
path_ch  = os.path.join(path_prefix, 'conjunto_de_datos_concentradohogar_enigh2022_ns.csv')
path_ing = os.path.join(path_prefix, 'conjunto_de_datos_ingresos_enigh2022_ns.csv')
path_pop = os.path.join(path_prefix, 'conjunto_de_datos_poblacion_enigh2022_ns.csv')

# 3. Columnas a extraer:
#    - Del concentrado de hogar: identificadores, geo, factor, tipo_localidad, y datos del jefe
ch_cols = [
    'folioviv', 'foliohog', 'ubica_geo', 'factor', 'tam_loc',
    'sexo_jefe', 'edad_jefe', 'educa_jefe'
]
#    - De ingresos: identifcadores y componentes de ingreso
ing_cols = [
    'folioviv', 'foliohog', 'numren',
    'ing_1','ing_2','ing_3','ing_4','ing_5','ing_6'
]
#    - De población: solo identificadores (ya no usaremos 'nivel')
pop_cols = ['folioviv', 'foliohog', 'numren', 'edad', 'sexo']

# 4. Carga de datos
ch  = pd.read_csv(path_ch,  usecols=ch_cols)
ing = pd.read_csv(path_ing, usecols=ing_cols)
pop = pd.read_csv(path_pop, usecols=pop_cols)

# 5. Renombrar columnas para consistencia
ch = ch.rename(columns={
    'ubica_geo': 'geo',
    'factor': 'factor_hogar',
    'tam_loc': 'tipo_localidad',
    'educa_jefe': 'escolaridad_jefe'
})
ing = ing.rename(columns={'numren': 'integrante'})
pop = pop.rename(columns={'numren': 'integrante'})

# 6. Merge de tablas
df = (
    ch
    .merge(ing, on=['folioviv','foliohog'], how='left')
    .merge(pop, on=['folioviv','foliohog','integrante'], how='left')
)

# 7. Limpieza de duplicados y nulos mínimos
df = df.drop_duplicates()
df['edad']      = df['edad'].fillna(-1).astype(int)
df['sexo']      = df['sexo'].fillna('N/D')
df['edad_jefe'] = df['edad_jefe'].fillna(-1).astype(int)
df['sexo_jefe'] = df['sexo_jefe'].fillna('N/D')

# 8. Procesamiento de ingresos
for c in ['ing_1','ing_2','ing_3','ing_4','ing_5','ing_6']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df[['ing_1','ing_2','ing_3','ing_4','ing_5','ing_6']] = df[['ing_1','ing_2','ing_3','ing_4','ing_5','ing_6']].fillna(0)
df['ingreso_nominal'] = df[['ing_1','ing_2','ing_3','ing_4','ing_5','ing_6']].sum(axis=1)
df = df[df['ingreso_nominal'] > 0]

# 9. Extraer entidad y mapear nombre de estado
df['geo']     = df['geo'].astype(str).str.zfill(4)
df['entidad'] = df['geo'].str[:2].astype(int)
map_ent = {
    1:'Aguascalientes',2:'Baja California',3:'Baja California Sur',4:'Campeche',
    5:'Coahuila',6:'Colima',7:'Chiapas',8:'Chihuahua',9:'Ciudad de México',
    10:'Durango',11:'Guanajuato',12:'Guerrero',13:'Hidalgo',14:'Jalisco',
    15:'México',16:'Michoacán',17:'Morelos',18:'Nayarit',19:'Nuevo León',
    20:'Oaxaca',21:'Puebla',22:'Querétaro',23:'Quintana Roo',
    24:'San Luis Potosí',25:'Sinaloa',26:'Sonora',27:'Tabasco',28:'Tamaulipas',
    29:'Tlaxcala',30:'Veracruz',31:'Yucatán',32:'Zacatecas'
}
df['estado'] = df['entidad'].map(map_ent)

# 10. Guardar resultado final
output = os.path.join(path_prefix, 'enigh2022_final.csv')
df.to_csv(output, index=False)

# 11. Vista previa
print("✔ Archivo guardado en:", output)
print(df[['estado','factor_hogar','tipo_localidad','sexo_jefe','edad_jefe','escolaridad_jefe']].drop_duplicates().head())


<ipython-input-9-5016577a222a>:50: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['edad']      = df['edad'].fillna(-1).astype(int)


✔ Archivo guardado en: /content/enigh2022_final.csv
     estado  factor_hogar  tipo_localidad  sexo_jefe  edad_jefe  \
0   Durango           206               1          2         91   
5   Durango           206               1          1         68   
12  Durango           206               1          1         56   
19  Durango           167               1          1         87   
22  Durango           167               1          1         27   

    escolaridad_jefe  
0                  3  
5                  8  
12                10  
19                11  
22                 8  
